# Задание

В этом задании:

1. Сделаем регрессию над данными через scikit-learn: сначала через регресию, потом через бустинг.
2. Сравним результаты с константным предсказанием.
3. Сделаем нейронную сеть на полносвязных слоях, обучим над теми же данными - и сравним с лин. регрессией и бустингом.



In [151]:
import pandas as pd
import numpy as np
import random

# Для воспроизводимости
seed = 0
np.random.seed(seed)
random.seed(seed)

In [152]:
df = pd.read_csv("insurance.csv")

### Задание №1:
Cделайте train/test split на данных в пропорции 0.8/0.2, залейте в лмс код, который в `df_train`, `df_test`
сохранит датафрейм с тренировочными и тестовыми данными соответственно.

P.S Использовать train_test_split из scikit-learn запрещено - разбивайте вручную через индексы.

In [153]:
index = np.arange(len(df))
np.random.shuffle(index)
df_train = df.iloc[index[:int(len(df) * 0.8)]]
df_test = df.iloc[index[int(len(df) * 0.8):]]

### Задание №2:
Выполните one-hot кодирование для категориальных признаков ``sex, region, smoker`` как в обучающем (``df_train``), так и в тестовом (``df_test``) датасетах.   
После кодирования указанные признаки должны быть заменены на соответствующие бинарные столбцы. Остальные признаки датафреймов должны остаться без изменений.


In [154]:
df_train = pd.get_dummies(df_train)
df_test = pd.get_dummies(df_test)

print(df_train.columns)
print(df_train.columns)

Index(['age', 'bmi', 'children', 'charges', 'sex_female', 'sex_male',
       'smoker_no', 'smoker_yes', 'region_northeast', 'region_northwest',
       'region_southeast', 'region_southwest'],
      dtype='str')
Index(['age', 'bmi', 'children', 'charges', 'sex_female', 'sex_male',
       'smoker_no', 'smoker_yes', 'region_northeast', 'region_northwest',
       'region_southeast', 'region_southwest'],
      dtype='str')


### Задание №4:
Нормализуйте колонки, которые вы отметили в квизе.
Считайте, что исходные датафреймы сохранены в `df_train` и `df_test`.

Сдайте код, который модифицирует `df_train` и `df_test` так, чтобы численные колонки из прошлого пункта стали нормированы.

In [155]:
for i in ["age", "bmi", "charges"]:
    df_train[i] = (df_train[i] - df_train[i].mean()) / df_train[i].std()
    df_test[i] = (df_test[i] - df_test[i].mean()) / df_test[i].std()

y_train = df_train.pop("charges").values.reshape(-1, 1)
y_test = df_test.pop("charges").values.reshape(-1, 1)

### Задание №5
Реализуйте функцию, считающую `MSE` метрику.

Ваша функция должна уметь принимать torch.Tensor, numpy-массивы и pd.Series.

In [156]:
from torch import Tensor, from_numpy, mean
def metric(preds, y):
    def to_torch(x):
        if isinstance(x, Tensor):
            return x
        if isinstance(x, pd.Series):
            return from_numpy(x.values)
        return from_numpy(np.array(x))

    preds_t = to_torch(preds)
    y_t = to_torch(y)

    return mean((y_t - preds_t) ** 2)

### Задание №6
Реализуйте бейзлайн на `LinearRegression` и `GradientBoostingRegressor`, отправьте метрики на _тестовой выборке_ в ЛМС.

Используйте гиперпараметры по-умолчанию в обоих моделях.

In [157]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression

base_gradient_model = GradientBoostingRegressor()
base_gradient_model.fit(df_train, y_train)
y_pred = base_gradient_model.predict(df_test)
mse_gb = metric(y_test, y_pred)
print("GradientBoostingRegressor MSE", mse_gb)

base_linear_model = LinearRegression()
base_linear_model.fit(df_train, y_train)
y_pred = base_linear_model.predict(df_test)
mse_l = metric(y_test, y_pred)
print("LinearRegression MSE", mse_l)

GradientBoostingRegressor MSE tensor(1.9463, dtype=torch.float64)
LinearRegression MSE tensor(0.2904, dtype=torch.float64)


/home/olesteep/Downloads/AI/lab1/venv/lib/python3.13/site-packages/sklearn/ensemble/_gb.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)  # TODO: Is this still required?


### Задание №7
Вычислите среднее значение целевой переменной на тренировочной выборке (train).

Подсчитайте MSE при константном предсказании этим средним и отправьте его в ЛМС.

In [158]:
y_mean = [sum(y_train)/len(y_train)] * len(y_train)

mse = metric(y_train, y_mean)

print(mse)

tensor(0.9991, dtype=torch.float64)


### Задание №8
Создайте сеть, состоящую из одного слоя Linear, залейте в лмс код, который в `model` запишет вашу модель

В качестве признаков используйте все колонки в текущем датасете, за исключением таргета

In [159]:
import torch.nn as nn
def build_model():

    model = nn.Linear(in_features=len(df_train.columns), out_features=1)

    # Модель должна иметь атрибуты weight, bias
    return model

model = build_model()

In [160]:
import torch

torch.random.manual_seed(seed)
t_x_train = torch.from_numpy(df_train.to_numpy().astype(float)).to(dtype=torch.float32)
t_y_train = torch.from_numpy(y_train).to(dtype=torch.float32)

print(t_x_train.shape)
print(t_y_train.shape)

torch.Size([1070, 11])
torch.Size([1070, 1])


### Задание №9
Напишите функцию `train_loop`, которая будет учить модель по данным на 2к итераций.
Считайте, что данные уже хранятся в переменных `t_x_train`, `t_y_train`.
Ваша функция `train_loop` должна вернуть список из лоссов на каждой итерации (т.е. список длины 2000).

Используйте `learning_rate=1e-2` в оптимизаторе.

Для простоты за одну итерацию делайте проход вперед и проход назад на всех наших обучающих данных.
Это будет полный градиентный спуск (не по батчам) - можем себе позволить, данных немного.

_Подсказка 1_: Вам не обязательно учить модель на видеокарте, CPU будет достаточно.

_Подсказка 2_: `tqdm` - это библиотека, которая рисует прогресс итераций

In [ ]:
import tqdm
from torch.optim.sgd import SGD

def train_loop(model: nn.Module) -> list[float]:
    losses = []
    criterion = nn.MSELoss()
    optimizer = SGD(params=model.parameters(), lr=1e-2)
    
    for _ in tqdm.trange(2000):
        optimizer.zero_grad()

        output = model(t_x_train)
        loss = criterion(output, t_y_train)
        loss.backward()

        optimizer.step()

        losses.append(loss.detach().item())
   
    return losses

### Задание №10
Обучите модель, состоящую из одного слоя `Linear`.
Приложите в ЛМС метрику `MSE` на тестовых данных.
Используйте `learning_rate=1e-2` в оптимизаторе.

Когда будете тестировать, не забудьте перенести тестовые данные в `torch.Tensor`

In [162]:
train_loop(model)
model.eval()

t_x_test = torch.from_numpy(df_test.to_numpy().astype(float)).to(dtype=torch.float32)
t_y_test = torch.from_numpy(y_test).to(dtype=torch.float32)

pred = model(t_x_test)
print(nn.MSELoss()(pred, t_y_test))

100%|██████████| 2000/2000 [00:00<00:00, 4770.08it/s]

tensor(0.2904, grad_fn=<MseLossBackward0>)


### Задание №11
Вам необходимо усложнить существующую нейронную сеть, добавив один скрытый слой.

Используйте следующие параметры:

Размерность скрытого слоя: 6 нейронов, функция активации -  `ReLU`

Приложите в лмс код, который в переменную `model` запишет вашу модель

In [163]:
def build_model():
    return nn.Sequential(
        nn.Linear(in_features=len(df_train.columns), out_features=6),
        nn.ReLU(),
        nn.Linear(in_features=6, out_features=1)
    )

new_model = build_model()

### Задание №12
Приложите в ЛМС метрику качества этой сети после 2к итераций обучения.
Эту модель можно обучить на CPU, не обязательно на видеокарте.

Используйте для обучения ту же функцию `train_loop` с теми же параметрами (`learning rate`, число итераций и т.п.)

In [164]:
train_loop(new_model)
new_model.eval()

pred = new_model(t_x_test)
print(nn.MSELoss()(pred, t_y_test))

100%|██████████| 2000/2000 [00:00<00:00, 3083.69it/s]

tensor(0.2240, grad_fn=<MseLossBackward0>)


### Задание №13
Добавьте дополнительные слои в нейронную сеть

Вам необходимо усложнить нейронную сеть, добавив еще 2-3 скрытых слоя с такими же размерностями, как в предыдущем задании.

Приложите в лмс код, который в переменную `model` запишет вашу модель

In [165]:
def build_model():
    return nn.Sequential(
        nn.Linear(in_features=len(df_train.columns), out_features=6),
        nn.ReLU(),
        nn.Linear(in_features=6, out_features=6),
        nn.ReLU(),
        nn.Linear(in_features=6, out_features=1)
    )

really_new_model = build_model()

### Задание №14
Приложите в ЛМС метрику качества после 2к итераций обучения.

Используйте для обучения ту же функцию `train_loop` с теми же параметрами (`learning rate`, число итераций и т.п.)

Эту модель можно обучить на CPU, не обязательно на видеокарте.

In [166]:
train_loop(really_new_model)

really_new_model.eval()

pred = really_new_model(t_x_test)
print(nn.MSELoss()(pred, t_y_test))

100%|██████████| 2000/2000 [00:00<00:00, 2531.14it/s]

tensor(0.2405, grad_fn=<MseLossBackward0>)
